# XGBoost — A Scalable Tree Boosting System (2016)

_Papers · Classical ML_

**Boosting with a second-order Taylor objective and a regularized, closed-form split score that made gradient boosting fast, accurate, and the go-to Kaggle workhorse.**

---

This notebook is a practice scaffold for the **XGBoost — A Scalable Tree Boosting System (2016)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — Python (NumPy + xgboost)

In [ ]:
# Setup: xgboost is not preinstalled in Colab -> install it (numpy already is).
# (Safe to re-run; no-op if already present.)
try:
    import xgboost  # noqa
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"], check=True)

import numpy as np
np.set_printoptions(precision=4, suppress=True)

# ---------------------------------------------------------------------------
# 0. The lesson's worked example, by hand. Squared-error loss => h_i = 1,
#    g_i = yhat - y. Current prediction yhat = 0 for everyone.
# ---------------------------------------------------------------------------
y = np.array([2.0, 3.0, -1.0, -2.0])      # labels, sorted by the one feature
g = 0.0 - y                                # gradient g_i = yhat - y  = [-2,-3,1,2]
h = np.ones_like(y)                        # hessian h_i = 1
lam, gamma = 1.0, 0.0

def leaf_score(G, H):     return G * G / (H + lam)        # G^2 / (H + lambda)
def leaf_value(G, H):     return -G / (H + lam)           # Eq. 5:  w* = -G/(H+lambda)

# Split "after example 2": left = first two, right = last two.
G_L, H_L = g[:2].sum(), h[:2].sum()        # -5, 2
G_R, H_R = g[2:].sum(), h[2:].sum()        #  3, 2
G,   H   = g.sum(),     h.sum()            # -2, 4

gain = 0.5 * (leaf_score(G_L, H_L) + leaf_score(G_R, H_R) - leaf_score(G, H)) - gamma  # Eq. 7
print("parent G,H =", (G, H), " leaf score =", round(leaf_score(G, H), 4))
print("left  G,H =", (G_L, H_L), " right G,H =", (G_R, H_R))
print("GAIN of split-after-2 =", round(gain, 4))                 # ~ 5.2667
print("leaf values:  w_L* =", round(leaf_value(G_L, H_L), 4),
      " w_R* =", round(leaf_value(G_R, H_R), 4))                 # 1.6667, -1.0

# ---------------------------------------------------------------------------
# 1. Greedy root split: sweep EVERY threshold of the feature, score each.
# ---------------------------------------------------------------------------
def best_split(g, h, lam=1.0, gamma=0.0):
    G_tot, H_tot = g.sum(), h.sum()
    best = (-np.inf, None)
    GL = HL = 0.0
    for k in range(1, len(g)):                 # split between sorted positions k-1 and k
        GL += g[k - 1]; HL += h[k - 1]
        GR, HR = G_tot - GL, H_tot - HL
        gain = 0.5 * (GL*GL/(HL+lam) + GR*GR/(HR+lam) - G_tot*G_tot/(H_tot+lam)) - gamma
        if gain > best[0]:
            best = (gain, k)
    return best

best_gain, k = best_split(g, h, lam, gamma)
print("\nbest split position =", k, " best gain =", round(best_gain, 4))  # k=2, ~5.2667

# ---------------------------------------------------------------------------
# 2. ABLATION: regularization lambda. Larger lambda -> smaller leaf values & gain.
# ---------------------------------------------------------------------------
print("\nlambda  |  w_L*    w_R*   |  gain")
for lam_ab in [0.0, 1.0, 10.0, 100.0]:
    wL = -G_L / (H_L + lam_ab); wR = -G_R / (H_R + lam_ab)
    gn = 0.5 * (G_L*G_L/(H_L+lam_ab) + G_R*G_R/(H_R+lam_ab)
                - G*G/(H+lam_ab))               # gamma = 0
    print(f"{lam_ab:6.1f}  | {wL:6.3f}  {wR:6.3f}  | {gn:6.3f}")
# leaf values shrink toward 0 and the gain falls as lambda rises -> regularization.

# ---------------------------------------------------------------------------
# 3. Confirm with the real xgboost library on the SAME toy data.
#    One tree, depth 1, matched reg_lambda/gamma, base_score=0.
# ---------------------------------------------------------------------------
import xgboost as xgb
X = np.array([[0.0], [1.0], [2.0], [3.0]])     # one feature, already sorted
dtrain = xgb.DMatrix(X, label=y)
params = dict(objective="reg:squarederror", max_depth=1, eta=1.0,
              reg_lambda=lam, gamma=gamma, base_score=0.0,
              min_child_weight=0, tree_method="exact")
bst = xgb.train(params, dtrain, num_boost_round=1)
print("\nxgboost dumped tree:\n", bst.get_dump()[0])
print("xgboost first-tree predictions:", bst.predict(dtrain))
# The dumped tree splits the feature between the +/- labels; its leaf values match
# our w* = -G/(H+lambda) up to xgboost's internal eta=1.0 scaling. Same split, same idea.

## Visualize the data & results

_As regularization lambda grows, what happens to the best split's gain and to the optimal leaf values?_

In [ ]:
import numpy as np
# Reproduce the lambda ablation on the worked-example split (g, h fixed).
g = np.array([-2.0, -3.0, 1.0, 2.0]); h = np.ones(4)
G_L, H_L = g[:2].sum(), h[:2].sum()      # -5, 2
G_R, H_R = g[2:].sum(), h[2:].sum()      #  3, 2
G,   H   = g.sum(),     h.sum()          # -2, 4
for lam in [0, 0.5, 1, 2, 5, 10, 20, 50, 100]:
    gain = 0.5 * (G_L*G_L/(H_L+lam) + G_R*G_R/(H_R+lam) - G*G/(H+lam))   # gamma = 0
    wL, wR = -G_L/(H_L+lam), -G_R/(H_R+lam)
    print(f"lam={lam:5}: gain={gain:.4f}  w_L*={wL:.4f}  w_R*={wR:.4f}")
# gain and |leaf values| both shrink monotonically as lambda grows -> regularization.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The regularization ablation. Using the worked example's four examples
            ($g=[-2,-3,1,2]$, $h=[1,1,1,1]$), recompute the gain of the "split after example 2" and the two
            optimal leaf values for $\lambda=1$ and then for $\lambda=10$ (with $\gamma=0$). What happens to
            the leaf magnitudes and to the gain as $\lambda$ grows, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Leaf sums are unchanged: $G_L=-5,H_L=2$; $G_R=3,H_R=2$; parent $G=-2,H=4$. — _$\lambda$ does not touch the gradient/hessian sums &mdash; it only enters the denominator $H+\lambda$._
- $\lambda=1$: $w_L^\ast=5/3\approx1.667$, $w_R^\ast=-1.0$; gain $=\tfrac12[25/3+9/3-4/5]\approx5.27$. — _Eq. 5 and Eq. 7 with $\lambda=1$ &mdash; the values are large and the split is very valuable._
- $\lambda=10$: $w_L^\ast=5/12\approx0.417$, $w_R^\ast=-3/12=-0.25$; gain $=\tfrac12[25/12+9/12-4/14]\approx1.27$. — _Bigger denominators shrink both the leaf values and every $G^2/(H+\lambda)$ term, so the gain drops too._

**Answer:** Raising $\lambda$ from $1$ to $10$ shrinks the leaf values toward 0 ($1.667\to0.417$,
                 $-1.0\to-0.25$) and lowers the gain ($\approx5.27\to\approx1.27$). Both follow from
                 $\lambda$ sitting in the denominator $H+\lambda$: it penalizes large leaf values and makes
                 splits that rest on few examples (small $H$) less attractive. That is exactly the bias-variance
                 knob &mdash; more $\lambda$ = simpler, more conservative trees.

</details>

**Problem 2.** Why does XGBoost need the hessian $h_i$ at all, when classical gradient boosting gets by with
            only the gradient $g_i$? Give the concrete role $h_i$ plays in the leaf value and the gain.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Look at the leaf value $w_j^\ast=-G_j/(H_j+\lambda)$ &mdash; the hessian sum $H_j$ sets the step size. — _Curvature tells you how sharply the loss bends, so $H_j$ scales how far the leaf can safely move; first-order boosting must guess this via a line search._
- Look at the gain $G^2/(H+\lambda)$ &mdash; $H$ weights how trustworthy a region is. — _Examples with small curvature (confident, in log-loss) contribute little $h$, so the optimal value and gain are computed in closed form per leaf instead of by a separate line search._
- Note that for squared error $h_i=1$, so the two coincide. — _That is why first-order least-squares boosting is the special case $h_i=1$; the hessian only earns its keep for losses with varying curvature like log-loss._

**Answer:** The hessian supplies the curvature, which lets XGBoost compute the best leaf value
                 $w^\ast=-G/(H+\lambda)$ and the split gain $G^2/(H+\lambda)$ in closed form, with the
                 right step size per region &mdash; no separate line search. For squared-error loss $h_i=1$ and it
                 reduces to ordinary residual boosting; for log-loss $h_i=p(1-p)$ varies, so confident points get
                 smaller, more cautious leaf updates. The hessian is what makes the second-order objective both
                 exact and fast.

</details>

**Problem 3.** You scan a feature and find the best split has gain $-0.4$ with $\gamma=0$ (and $0.5$ with
            $\gamma=1$). Do you split? What does the answer say about the role of $\gamma$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the rule: split only if $\mathcal{L}_{\text{split}}\gt0$ (the gain already has $-\gamma$ baked in). — _Eq. 7 ends in $-\gamma$; a non-positive gain means the split does not improve the regularized objective enough to justify a new leaf._
- Case $\gamma=0$: gain $=-0.4\lt0$ &rarr; do not split. — _Even free of the per-leaf cost, the bracket itself is negative &mdash; the two groups are not separable enough._
- Case $\gamma=1$: a reported gain of $0.5$ means the bracket was $1.5$ before subtracting $\gamma=1$. $1.5-1=0.5\gt0$ &rarr; split. — _Same data, different $\gamma$: a smaller $\gamma$ keeps more splits, a larger one prunes them &mdash; the per-leaf complexity cost._

**Answer:** With $\gamma=0$ the gain is $-0.4\lt0$, so you do not split. With $\gamma=1$ the
                 (different) reported gain of $0.5\gt0$, so you do. $\gamma$ is the minimum improvement a
                 split must buy to be worth an extra leaf: it is the pruning knob built directly into the gain
                 formula, so XGBoost prunes while growing rather than afterward.

</details>